# Microproyecto 2: Generación de Código con Transformers

**Curso:** Modelos Avanzados para el Procesamiento de Lenguaje Natural
**Equipo 4**
**Integrantes:** Juan Sebastián Páez Cortés, Julian David Parada Galvis, Luis Carlos Prada Socha, Jean Pierre Betancourt Padilla
**Fecha:** octubre de 2026

---

## 1. Portada e introducción

### Resumen

En este microproyecto entrenamos modelos Transformer decoder-only para generar
código en Python. El modelo aprende a predecir el siguiente token de un archivo de código y,
con eso, puede continuar un fragmento que le demos como entrada.

Comparamos cinco configuraciones para ver cuánto depende el resultado del tamaño del modelo
y de si se entrena desde cero o se parte de pesos ya preentrenados:

| # | Configuración | Modelo | Entrenamiento |
| :-: | :--- | :--- | :--- |
| 1 | Desde cero | Transformer decoder-only en PyTorch | 10 épocas desde pesos aleatorios |
| 2 | Baseline 135M | SmolLM2-135M | Sin entrenar |
| 3 | Finetuning 135M | SmolLM2-135M | Full finetuning, 2 épocas |
| 4 | Baseline 360M | SmolLM2-360M | Sin entrenar |
| 5 | Finetuning 360M | SmolLM2-360M | Full finetuning, 2 épocas |

### Datos

Usamos `codeparrot/codeparrot-clean` en modo streaming. Nos quedamos solo con archivos de
entre 1.000 y 2.000 caracteres, tomamos 8.000 ejemplos y los dividimos en 6.400 de
entrenamiento, 800 de validación y 800 de prueba como lo pide el proyecto. Todos los modelos
usan el tokenizador de SmolLM2 (49.152 tokens) y un contexto de 256 tokens. El filtro, el split
y el contexto son los que exige la guía del proyecto.

### Evaluación

- Perplejidad sobre los 800 ejemplos de prueba.
- Porcentaje de código sintácticamente válido con `ast` en 200 ejemplos.
- Tres ejemplos por cada modelo entrenado, con nuestro propio bucle autorregresivo.

### Entorno

El notebook corre completo en Google Colab con GPU L4. Para el modelo de 360M usamos
precisión mixta, acumulación de gradientes y gradient checkpointing, que es la estrategia de
memoria que recomienda el personal del curso. Entre un modelo y el siguiente liberamos la
memoria de la GPU.

Todo el pipeline es reproducible: fijamos la semilla y la revisión del dataset y del
tokenizador, de modo que dos ejecuciones producen los mismos 8.000 archivos y el mismo split.


## 2. Librerías

Dejamos listo el entorno de ejecución antes de tocar los datos, siguiendo lo que pide la guía
del proyecto.


Instalamos las versiones de las librerías de Hugging Face que se usan en el proyecto.


In [ ]:
!pip install -q "transformers>=4.44,<5" "datasets>=2.20,<4" "accelerate>=0.33"


Importamos lo necesario y comprobamos las versiones cargadas; `ast` lo usaremos para la
validación sintáctica que pide la guía.


In [ ]:
import ast
import gc
import json
import math
import os
import random
import shutil
import time
import warnings

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"  # workspace fijo para que las multiplicaciones de matrices den siempre lo mismo

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

import datasets
import transformers
from datasets import load_dataset
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer

import matplotlib.pyplot as plt
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"  # evita problemas con los avisos de fork en Colab

print("Entorno cargado")
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("numpy:", np.__version__)


Configuramos la GPU, la precisión mixta, las constantes que fija el enunciado (tokenizador
SmolLM2, contexto 256 y batch efectivo 16) y la carpeta de checkpoints.


In [ ]:
SEED = 42

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_BF16 = DEVICE == "cuda" and torch.cuda.is_bf16_supported()

if USE_BF16:  # elige precision mixta segun la GPU de Colab (T4 o L4)
    AMP_DTYPE = torch.bfloat16
    NOMBRE_AMP = "bfloat16"
    USE_SCALER = False
else:
    AMP_DTYPE = torch.float16
    NOMBRE_AMP = "float16"
    USE_SCALER = True

if DEVICE == "cuda":
    props = torch.cuda.get_device_properties(0)
    print("GPU:", props.name)
    print("VRAM:", round(props.total_memory / 1024**3, 1), "GB")
else:
    print("No se detecto GPU, el entrenamiento sera con CPU y puede ser muy lento")

print("AMP:", NOMBRE_AMP)
print("GradScaler:", USE_SCALER)

SMOL_135M = "HuggingFaceTB/SmolLM2-135M"
SMOL_360M = "HuggingFaceTB/SmolLM2-360M"
TOKENIZER_ID = SMOL_135M  # uno solo para las cinco configuraciones porque la PPL se mide por token

DATASET_ID = "codeparrot/codeparrot-clean"
DATASET_REVISION = "35a59fb025bc0a102f7d96eac09d145b896d487b"
SMOL_135M_REVISION = "93efa2f097d58c2a74874c7e644dbc9b0cee75a2"
SMOL_360M_REVISION = "f8027fd0eaeea54caa13c31d31b9fdc459c38b49"

CTX = 256
VOCAB_SIZE = 49152

MIN_CHARS = 1000
MAX_CHARS = 2000
N_TOTAL = 8000
N_TRAIN = 6400
N_VAL = 800
N_TEST = 800

BATCH_EFECTIVO = 16
CLIP_GRAD_NORM = 1.0

drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/microproyecto_2"
os.makedirs(DRIVE_DIR, exist_ok=True)

print("Checkpoints:", DRIVE_DIR)

if "HF_TOKEN" in os.environ:
    print("HF_TOKEN, configurado")
else:
    print("HF_TOKEN: no esta en las variables de entorno")


Fijamos la semilla para que los resultados sean reproducibles y creamos la función que libera
la GPU entre modelos, como indica el documento de checkpoints.


In [ ]:
def fijar_semilla(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)


def liberar_vram_total(modelo=None, optimizador=None):
    if modelo is not None:
        del modelo
    if optimizador is not None:
        del optimizador
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        torch.cuda.reset_peak_memory_stats()


fijar_semilla()
print("Semilla:", SEED)


## 3. Carga del dataset y exploración

Trabajamos con `codeparrot/codeparrot-clean`, el dataset que indica el proyecto. Lo cargamos en
modo streaming para no descargar los 5 GB completos: los ejemplos se van pidiendo a medida que
los necesitamos.

Primero miramos la estructura del dataset, después qué tan largos son los archivos (el
enunciado solo acepta los que tienen entre 1.000 y 2.000 caracteres) y, por último, revisamos
licencias, repos y duplicados para saber con qué datos estamos trabajando.


Cargamos el dataset en streaming, fijando la revisión para que el contenido no cambie entre
ejecuciones, y miramos qué columnas trae.


In [ ]:
dataset = load_dataset(
    DATASET_ID,
    split="train",
    streaming=True,
    revision=DATASET_REVISION,
)
ejemplo = next(iter(dataset))

print(dataset)
print("Columnas:", list(ejemplo.keys()))
print("Revision:", DATASET_REVISION)


Mostramos un archivo de ejemplo para ver cómo viene el campo `content`.


In [ ]:
print("repo:", ejemplo["repo_name"])
print("archivo:", ejemplo["path"])
print("caracteres:", len(ejemplo["content"]))
print("lineas:", ejemplo["content"].count("\n"))
print()
print(ejemplo["content"][:600])


Medimos 2.000 archivos y guardamos la muestra completa para analizarla sin volver al stream.


In [ ]:
N_MUESTRA = 2000
muestra = []

for i, ejemplo in enumerate(dataset):
    if i == N_MUESTRA:
        break
    muestra.append(ejemplo)

longitudes = []
for e in muestra:
    longitudes.append(len(e["content"]))

print("Archivos medidos:", len(muestra))


Contamos cuántos archivos caen en el rango que pide el enunciado y resumimos la distribución.


In [ ]:
dentro_rango = 0
for largo in longitudes:
    if largo >= MIN_CHARS and largo <= MAX_CHARS:
        dentro_rango += 1

ordenadas = sorted(longitudes)
n = len(ordenadas)

print("En el rango:", dentro_rango, "de", n)
print("Porcentaje:", round(100 * dentro_rango / n, 1), "%")
print("Minimo:", ordenadas[0])
print("P25:", ordenadas[n // 4])
print("Mediana:", ordenadas[n // 2])
print("P75:", ordenadas[3 * n // 4])
print("P90:", ordenadas[9 * n // 10])
print("P99:", ordenadas[99 * n // 100])
print("Maximo:", ordenadas[-1])
print("Media:", round(sum(longitudes) / n, 1))


Graficamos la distribución completa de longitudes.


In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(longitudes, bins=60)
plt.axvline(MIN_CHARS, color="red", linestyle="--")
plt.axvline(MAX_CHARS, color="red", linestyle="--")
plt.title("Longitud de los archivos de codeparrot-clean")
plt.xlabel("caracteres")
plt.ylabel("cantidad de archivos")
plt.show()


La gráfica anterior muestra la distribución completa, pero la cola de archivos muy largos
aplasta la zona que nos interesa. Por eso repetimos el histograma recortado a 4.000 caracteres:
no estamos cambiando los datos, solo el zoom de la figura.


In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(longitudes, bins=60, range=(0, 4000))
plt.axvline(MIN_CHARS, color="red", linestyle="--")
plt.axvline(MAX_CHARS, color="red", linestyle="--")
plt.title("Longitud de los archivos (vista hasta 4.000 caracteres)")
plt.xlabel("caracteres")
plt.ylabel("cantidad de archivos")
plt.xlim(0, 4000)
plt.show()


Revisamos el tipo de dato y un valor de ejemplo de cada columna.


In [ ]:
print("Columna           Tipo       Ejemplo")
for nombre in muestra[0]:
    valor = muestra[0][nombre]
    texto = str(valor).replace("\n", " ")
    print(nombre.ljust(17), type(valor).__name__.ljust(10), texto[:45])


Revisamos si hay archivos vacíos o con contenido nulo.


In [ ]:
vacios = 0
solo_espacios = 0

for e in muestra:
    contenido = e["content"]
    if contenido is None or len(contenido) == 0:
        vacios += 1
    elif len(contenido.strip()) == 0:
        solo_espacios += 1

print("Vacios o nulos:", vacios)
print("Solo espacios:", solo_espacios)


Comparamos el archivo más corto de la muestra, uno que cae en el rango y uno largo. Ojo: casi
no aparecen archivos de menos de 1.000 caracteres, así que el límite que de verdad descarta es
el de 2.000.


In [ ]:
menores_1000 = 0
for e in muestra:
    if len(e["content"]) < MIN_CHARS:
        menores_1000 += 1

print("Archivos con menos de 1.000 caracteres:", menores_1000)

mas_corto = muestra[0]
en_rango = None
largo = None

for e in muestra:
    n = len(e["content"])
    if n < len(mas_corto["content"]):
        mas_corto = e
    if en_rango is None and n >= MIN_CHARS and n <= MAX_CHARS:
        en_rango = e
    if largo is None and n > MAX_CHARS:
        largo = e

casos = [("MAS CORTO", mas_corto), ("EN RANGO", en_rango), ("LARGO", largo)]

for nombre, e in casos:
    if e is None:
        print("=====", nombre, "- no se encontro en la muestra =====")
        print()
        continue
    print("=====", nombre, "-", len(e["content"]), "caracteres =====")
    print(e["content"][:300])
    print()


Contamos cuántas veces aparece cada construcción típica de Python.


In [ ]:
palabras = ["def ", "class ", "import ", "return ", "print("]
conteos = {}

for palabra in palabras:
    conteos[palabra] = 0

for e in muestra:
    for palabra in palabras:
        conteos[palabra] += e["content"].count(palabra)

valores = []
for palabra in palabras:
    print(palabra.strip(), conteos[palabra])
    valores.append(conteos[palabra])

plt.figure(figsize=(8, 3))
plt.barh(palabras, valores)
plt.title("Construcciones de Python en 2.000 archivos")
plt.xlabel("apariciones")
plt.show()


Revisamos cuáles son las licencias más comunes.


In [ ]:
licencias = {}

for e in muestra:
    lic = str(e["license"])
    if lic in licencias:
        licencias[lic] += 1
    else:
        licencias[lic] = 1

pares = []
for lic, cantidad in licencias.items():
    pares.append((cantidad, lic))
pares.sort(reverse=True)

nombres = []
valores = []
for cantidad, lic in pares[:5]:
    print(cantidad, lic)
    nombres.append(lic[:30])
    valores.append(cantidad)

plt.figure(figsize=(8, 3))
plt.barh(nombres, valores)
plt.title("Licencias mas comunes")
plt.xlabel("archivos")
plt.show()


Revisamos si los archivos vienen de pocos repos y cuántos están repetidos.


In [ ]:
repos = {}

for e in muestra:
    repo = e["repo_name"]
    if repo in repos:
        repos[repo] += 1
    else:
        repos[repo] = 1

pares = []
for repo, cantidad in repos.items():
    pares.append((cantidad, repo))
pares.sort(reverse=True)

print("Repos distintos:", len(repos))
for cantidad, repo in pares[:5]:
    print(cantidad, repo)

vistos = set()
repetidos = 0
for e in muestra:
    if e["hash"] in vistos:
        repetidos += 1
    else:
        vistos.add(e["hash"])

print("Archivos repetidos por hash:", repetidos)


Estimamos cuántos archivos hay que revisar para juntar los 8.000 que pide el enunciado.


In [ ]:
# si 399 de cada 2.000 sirven, para 8.000 necesitamos 8000 x 2000 / 399
acumulado = []
total = 0

for largo in longitudes:
    if largo >= MIN_CHARS and largo <= MAX_CHARS:
        total += 1
    acumulado.append(total)

esperado = round(N_TOTAL * len(longitudes) / dentro_rango)

plt.figure(figsize=(8, 4))
plt.plot(acumulado, label="muestra")
plt.plot([0, esperado], [0, N_TOTAL], linestyle="--", label="proyeccion")
plt.axhline(N_TOTAL, color="red", linestyle="--")
plt.xlim(0, esperado * 1.05)
plt.ylim(0, N_TOTAL * 1.1)
plt.title("Archivos utiles acumulados")
plt.xlabel("archivos revisados")
plt.ylabel("archivos en rango")
plt.legend()
plt.show()

print("Para 8.000 utiles hay que revisar cerca de", esperado, "archivos")


Medimos cuántos archivos de la muestra son Python válido y cuántos traen sintaxis de Python 2.


In [ ]:
validos_muestra = 0
con_print_viejo = 0
con_except_viejo = 0

for e in muestra:
    codigo = e["content"]
    try:
        ast.parse(codigo)
        validos_muestra += 1
    except SyntaxError:
        pass
    if "print " in codigo:
        con_print_viejo += 1
    if ", e:" in codigo:
        con_except_viejo += 1

print("Archivos validos con ast:", validos_muestra, "de", len(muestra))
print("Con posible print de Python 2:", con_print_viejo)
print("Con posible except de Python 2:", con_except_viejo)


### Conclusiones de la exploración

**Solo 2 de cada 10 archivos sirven.** De los 2.000 que revisamos, 399 (19,9 %) caen entre
1.000 y 2.000 caracteres. Para juntar los 8.000 que pide el enunciado hay que recorrer unos
40.100 archivos, o sea cinco veces más de lo que costaría si todos calificaran. En tiempo no
duele: 2.000 archivos tardaron 1,4 s, así que el filtro completo toma unos 30 s.

**La distribución es de cola larga.** La media (10.263,6) es 2,2 veces la mediana (4.733). Si la
distribución fuera simétrica, media y mediana coincidirían; que estén tan separadas significa
que unos pocos archivos gigantes arrastran el promedio. El máximo es 456.237 caracteres: 96
veces la mediana y 445 veces el archivo más corto. Por eso la media no describe estos datos y
preferimos la mediana y los percentiles para hablar del "archivo típico".

**El límite inferior no filtra nada.** El archivo más corto de la muestra tiene 1.027 caracteres
y no apareció ni uno por debajo de 1.000: codeparrot-clean ya viene pre-filtrado. Todo el
descarte lo hace el techo de 2.000, que deja fuera a 8 de cada 10 archivos.

**El techo también sesga el contenido.** Además de limitar el tamaño, tiende a quedarse con
archivos de poca cabecera: los que traen bloques grandes de licencia o comentarios iniciales
suelen pasar los 2.000 caracteres y quedan fuera. Es una limitación del protocolo, no algo que
podamos corregir sin incumplirlo, pero conviene recordarla al interpretar los resultados.

**Qué clase de código esperar.** En 2.000 archivos contamos 27.325 `def`, 13.547 `import` y
17.149 `return`, contra solo 1.297 `print`. Domina la definición de funciones dentro de módulos,
no scripts que corren de principio a fin. Por eso esperamos que los modelos finetuneados generen
funciones y clases plausibles antes que programas completos, y que el modelo desde cero salga
mucho más plano y repetitivo.

**Calidad y procedencia.** No encontramos `content` vacíos, nulos ni duplicados por `hash`. Los
2.000 archivos vienen de 1.855 repos distintos (máximo 5 de un mismo repo), así que el split
aleatorio no debería mezclar el mismo proyecto entre train y test. Las licencias son en su
mayoría permisivas (apache 421, mit 366, bsd 347), con algo de copyleft (gpl-3 347, agpl-3 186);
como los bloques conservan las cabeceras, el modelo también aprende a emitirlas. Medimos además
cuántos archivos parsean con `ast` y cuántos traen señales de Python 2, porque eso pone el techo
de lo que la métrica de validez de la sección 8 puede llegar a medir.

En volumen, los 8.000 archivos filtrados dieron 3.460.082 tokens, que se empacan en 13.514
bloques de 256. Eso es lo que alimenta el entrenamiento de la sección 6.


## 4. Preprocesamiento

Aplicamos el filtro de longitud que pide el enunciado, tokenizamos con el tokenizador de
SmolLM2 y empacamos los tokens en bloques fijos de 256, que es el contexto que van a usar los
cinco modelos. El empaquetado definitivo se hace por split en la sección 5, para que ningún
bloque mezcle tokens de entrenamiento con tokens de prueba.


Cargamos el tokenizador de SmolLM2 con su revisión fija y revisamos su vocabulario.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_ID, revision=SMOL_135M_REVISION)
tokenizer_360 = AutoTokenizer.from_pretrained(SMOL_360M, revision=SMOL_360M_REVISION)

print("Tokenizador:", TOKENIZER_ID)
print("Vocabulario:", len(tokenizer))
print("Token de fin:", tokenizer.eos_token, "-", tokenizer.eos_token_id)
print("Revision:", SMOL_135M_REVISION)
print("Vocabularios iguales:", tokenizer.get_vocab() == tokenizer_360.get_vocab())

del tokenizer_360


Filtramos el stream y nos quedamos con los 8.000 primeros archivos del rango.


In [ ]:
textos = []

inicio = time.time()
for e in dataset:
    largo = len(e["content"])
    if largo >= MIN_CHARS and largo <= MAX_CHARS:
        textos.append(e["content"])
    if len(textos) == N_TOTAL:
        break

print("Archivos filtrados:", len(textos))
print("Tiempo:", round(time.time() - inicio, 1), "segundos")

if len(textos) == N_TOTAL:
    print("Se encontraron los", N_TOTAL, "archivos que pide el enunciado")
else:
    print("Se esperaban", N_TOTAL, "archivos y se encontraron", len(textos))


Tokenizamos cada archivo y le agregamos el token de fin de texto.


In [ ]:
ids_por_texto = []
total_tokens = 0
total_caracteres = 0

for texto in textos:
    ids = tokenizer.encode(texto, add_special_tokens=False)
    ids.append(tokenizer.eos_token_id)
    ids_por_texto.append(ids)
    total_tokens += len(ids)
    total_caracteres += len(texto)

print("Archivos tokenizados:", len(ids_por_texto))
print("Tokens en total:", total_tokens)
print("Promedio por archivo:", round(total_tokens / len(ids_por_texto), 1))
print("Caracteres por token:", round(total_caracteres / total_tokens, 2))


Mostramos cómo se ve un archivo después de tokenizarlo.


In [ ]:
primeros_ids = ids_por_texto[0][:40]
print("Primeros tokens:", primeros_ids)
print("Como texto:", tokenizer.decode(primeros_ids))


Empacamos los tokens en bloques fijos de 256 y probamos la función con 50 archivos.


In [ ]:
def empacar(lista_ids, contexto):
    bloques = []
    buffer = []
    for ids in lista_ids:
        buffer.extend(ids)
        while len(buffer) >= contexto:
            bloques.append(buffer[:contexto])
            buffer = buffer[contexto:]
    return bloques


bloques_prueba = empacar(ids_por_texto[:50], CTX)

print("Bloques de prueba:", len(bloques_prueba))
print("Largo de cada bloque:", len(bloques_prueba[0]))
print("Bloques aproximados con los 8.000 juntos:", total_tokens // CTX)
print()
print(tokenizer.decode(bloques_prueba[0][:80]))


## 5. Split train / validación / prueba

Partimos los 8.000 textos en 6.400 de entrenamiento, 800 de validación y 800 de prueba. Primero
mezclamos con la semilla fija para que el corte no dependa del orden en que llegan los archivos
del stream, y después empacamos cada parte por separado en bloques de 256 tokens.


Mezclamos los textos y sus tokens con la semilla fija.


In [ ]:
rng = random.Random(SEED)
orden = list(range(N_TOTAL))
rng.shuffle(orden)

textos_mezclados = []
ids_mezclados = []

for i in orden:
    textos_mezclados.append(textos[i])
    ids_mezclados.append(ids_por_texto[i])

print("Textos mezclados:", len(textos_mezclados))
print("Semilla:", SEED)


Partimos en 6.400 de entrenamiento, 800 de validación y 800 de prueba.


In [ ]:
textos_train = textos_mezclados[:N_TRAIN]
textos_val = textos_mezclados[N_TRAIN:N_TRAIN + N_VAL]
textos_test = textos_mezclados[N_TRAIN + N_VAL:]

ids_train = ids_mezclados[:N_TRAIN]
ids_val = ids_mezclados[N_TRAIN:N_TRAIN + N_VAL]
ids_test = ids_mezclados[N_TRAIN + N_VAL:]

print("Train:", len(textos_train))
print("Validacion:", len(textos_val))
print("Prueba:", len(textos_test))
print("Total:", len(textos_train) + len(textos_val) + len(textos_test))


Verificamos que ningún texto quede en dos partes a la vez.


In [ ]:
set_train = set(textos_train)
set_val = set(textos_val)
set_test = set(textos_test)

print("Repetidos train-validacion:", len(set_train & set_val))
print("Repetidos train-prueba:", len(set_train & set_test))
print("Repetidos validacion-prueba:", len(set_val & set_test))
print("Textos unicos:", len(set_train | set_val | set_test))


Empacamos cada parte por separado en bloques de 256 tokens.


In [ ]:
bloques_train = empacar(ids_train, CTX)
bloques_val = empacar(ids_val, CTX)
bloques_test = empacar(ids_test, CTX)

print("Bloques train:", len(bloques_train))
print("Bloques validacion:", len(bloques_val))
print("Bloques prueba:", len(bloques_test))
print("Bloques totales:", len(bloques_train) + len(bloques_val) + len(bloques_test))


Convertimos los bloques a tensores y revisamos sus formas.


In [ ]:
X_train = torch.tensor(bloques_train, dtype=torch.long)
X_val = torch.tensor(bloques_val, dtype=torch.long)
X_test = torch.tensor(bloques_test, dtype=torch.long)

print("Train:", X_train.shape, X_train.dtype)
print("Validacion:", X_val.shape, X_val.dtype)
print("Prueba:", X_test.shape, X_test.dtype)
print("Tokens de entrenamiento:", X_train.numel())


Revisamos cuántos tokens se pierden al final de cada parte por no completar un bloque.


In [ ]:
partes = [
    ("train", ids_train, bloques_train),
    ("validacion", ids_val, bloques_val),
    ("prueba", ids_test, bloques_test),
]

for nombre, ids, bloques in partes:
    total = 0
    for x in ids:
        total += len(x)
    usados = len(bloques) * CTX
    print(nombre, "- tokens:", total, "| usados:", usados, "| descartados:", total - usados)


## 6. Modelo desde cero

Implementamos un Transformer decoder-only en PyTorch puro: atención multi-cabeza con máscara
causal, bloques con LayerNorm y MLP, embeddings de token y de posición, y una cabeza de salida
atada a los embeddings. Tiene 6 capas, 6 cabezas y 384 dimensiones (unos 29,6 M de parámetros),
y se entrena desde pesos aleatorios.

Usamos 10 épocas en vez de las 3 que sugiere la tabla del enunciado: con 2,75 M tokens por
época el modelo probablemente quede con margen de mejora, y la curva de validación nos permitirá
ver ese comportamiento. Para compensar
subimos el dropout y guardamos el mejor modelo según validación.


Definimos los hiperparámetros del modelo y la máscara causal.


In [ ]:
N_LAYER = 6
N_HEAD = 6
N_EMBD = 384
N_MLP = 4 * N_EMBD
DROPOUT = 0.15

EPOCAS = 10
BATCH = 16
LR = 3e-4
LR_MIN = 3e-5
CALENTAMIENTO = 100
WEIGHT_DECAY = 0.1

mascara = torch.tril(torch.ones(CTX, CTX)).view(1, 1, CTX, CTX).to(DEVICE)

print("Capas:", N_LAYER, "| Cabezas:", N_HEAD, "| Dimension:", N_EMBD)
print("Dropout:", DROPOUT, "| Epocas:", EPOCAS, "| LR:", LR)


Implementamos la atención multi-cabeza con máscara.


In [ ]:
class CabezaAtencion(nn.Module):
    def __init__(self):
        super().__init__()
        self.c_attn = nn.Linear(N_EMBD, 3 * N_EMBD, bias=False)
        self.c_proy = nn.Linear(N_EMBD, N_EMBD, bias=False)
        self.dropout = nn.Dropout(DROPOUT)

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.c_attn(x).split(N_EMBD, dim=2)
        q = q.view(B, T, N_HEAD, C // N_HEAD).transpose(1, 2)
        k = k.view(B, T, N_HEAD, C // N_HEAD).transpose(1, 2)
        v = v.view(B, T, N_HEAD, C // N_HEAD).transpose(1, 2)

        pesos = q @ k.transpose(-2, -1)
        pesos = pesos / math.sqrt(C // N_HEAD)
        pesos = pesos.masked_fill(mascara[:, :, :T, :T] == 0, float("-inf"))
        pesos = F.softmax(pesos, dim=-1)
        pesos = self.dropout(pesos)

        salida = pesos @ v
        salida = salida.transpose(1, 2).contiguous().view(B, T, C)
        return self.c_proy(salida)


Armamos el bloque del Transformer con LayerNorm, atención, MLP y conexiones residuales.


In [ ]:
class Bloque(nn.Module):
    def __init__(self):
        super().__init__()
        self.ln1 = nn.LayerNorm(N_EMBD)
        self.atencion = CabezaAtencion()
        self.ln2 = nn.LayerNorm(N_EMBD)
        self.mlp = nn.Sequential(
            nn.Linear(N_EMBD, N_MLP, bias=False),
            nn.GELU(),
            nn.Linear(N_MLP, N_EMBD, bias=False),
        )
        self.dropout = nn.Dropout(DROPOUT)

    def forward(self, x):
        x = x + self.dropout(self.atencion(self.ln1(x)))
        x = x + self.dropout(self.mlp(self.ln2(x)))
        return x


Implementamos el modelo completo con embeddings de token y posición y la cabeza de salida.


In [ ]:
class ModeloInicial(nn.Module):
    def __init__(self):
        super().__init__()
        self.tok_emb = nn.Embedding(VOCAB_SIZE, N_EMBD)
        self.pos_emb = nn.Embedding(CTX, N_EMBD)
        self.dropout = nn.Dropout(DROPOUT)
        self.bloques = nn.ModuleList([Bloque() for _ in range(N_LAYER)])
        self.ln_f = nn.LayerNorm(N_EMBD)
        self.lm_head = nn.Linear(N_EMBD, VOCAB_SIZE, bias=False)
        self.lm_head.weight = self.tok_emb.weight

        self.apply(self.inicializar)
        for nombre, parametro in self.named_parameters():
            if nombre.endswith("c_proy.weight") or nombre.endswith("mlp.2.weight"):
                nn.init.normal_(parametro, mean=0.0, std=0.02 / math.sqrt(2 * N_LAYER))

    def inicializar(self, modulo):
        if isinstance(modulo, nn.Linear):
            nn.init.normal_(modulo.weight, mean=0.0, std=0.02)
        elif isinstance(modulo, nn.Embedding):
            nn.init.normal_(modulo.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        posiciones = torch.arange(T, device=idx.device)
        x = self.tok_emb(idx) + self.pos_emb(posiciones)
        x = self.dropout(x)

        for bloque in self.bloques:
            x = bloque(x)

        x = self.ln_f(x)
        logits = self.lm_head(x)

        perdida = None
        if targets is not None:
            logits_planos = logits[:, :-1, :].reshape(-1, VOCAB_SIZE)
            objetivos = targets[:, 1:].reshape(-1)
            perdida = F.cross_entropy(logits_planos, objetivos)

        return logits, perdida


Creamos el modelo en la GPU.


In [ ]:
modelo_inicial = ModeloInicial().to(DEVICE)

print("Modelo creado en:", DEVICE)


Preparamos los lotes de entrenamiento y de validación.


In [ ]:
datos_train = TensorDataset(X_train)
datos_val = TensorDataset(X_val)

generador = torch.Generator()
generador.manual_seed(SEED)

train_loader = DataLoader(datos_train, batch_size=BATCH, shuffle=True, generator=generador, drop_last=True)
val_loader = DataLoader(datos_val, batch_size=BATCH, shuffle=False)

print("train:", len(train_loader))
print("validacion:", len(val_loader))


Configuramos el optimizador.


In [ ]:
optimizer = torch.optim.AdamW(
    modelo_inicial.parameters(),
    lr=LR,
    betas=(0.9, 0.99),
    weight_decay=WEIGHT_DECAY,
)

PASOS_POR_EPOCA = len(train_loader)
PASOS_TOTALES = PASOS_POR_EPOCA * EPOCAS


def tasa_lr(paso):
    if paso < CALENTAMIENTO:
        return (paso + 1) / CALENTAMIENTO
    avance = (paso - CALENTAMIENTO) / (PASOS_TOTALES - CALENTAMIENTO)
    if avance > 1:
        avance = 1
    return LR_MIN / LR + (1 - LR_MIN / LR) * 0.5 * (1 + math.cos(math.pi * avance))


scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, tasa_lr)

print("Pasos por epoca:", PASOS_POR_EPOCA)
print("Pasos totales:", PASOS_TOTALES)


Entrenamos y si el checkpoint ya existe en Drive lo cargamos y saltamos el entrenamiento.


In [ ]:
ruta_scratch = os.path.join(DRIVE_DIR, "checkpoint_scratch.pt")
ruta_historial_scratch = os.path.join(DRIVE_DIR, "historial_scratch.json")

entreno_scratch = True
if os.path.exists(ruta_scratch):
    print("checkpoint existente, se omite el entrenamiento")
    modelo_inicial.load_state_dict(torch.load(ruta_scratch, map_location=DEVICE))
    entreno_scratch = False

historial = {"train": [], "val": []}
mejor_val = float("inf")
mejor_epoca = 0
mejor_estado = None

if entreno_scratch == False:
    if os.path.exists(ruta_historial_scratch):
        archivo = open(ruta_historial_scratch, "r")
        guardado = json.load(archivo)
        archivo.close()
        historial = guardado["historial"]
        mejor_val = guardado["mejor_val"]
        mejor_epoca = guardado["mejor_epoca"]
        print("historial recuperado de", ruta_historial_scratch)

if entreno_scratch:
    modelo_inicial.eval()
    suma0 = 0.0
    ejemplos0 = 0
    with torch.no_grad():
        for (lote,) in val_loader:
            lote = lote.to(DEVICE)
            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
                _, perdida = modelo_inicial(lote, lote)
            suma0 += perdida.item() * lote.size(0)
            ejemplos0 += lote.size(0)
    print("val inicial:", round(suma0 / ejemplos0, 3))

    for epoca in range(EPOCAS):
        modelo_inicial.train()
        suma = 0.0
        ejemplos = 0
        inicio = time.time()

        for (lote,) in train_loader:
            lote = lote.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
                _, perdida = modelo_inicial(lote, lote)
            perdida.backward()
            torch.nn.utils.clip_grad_norm_(modelo_inicial.parameters(), CLIP_GRAD_NORM)
            optimizer.step()
            scheduler.step()
            suma += perdida.item() * lote.size(0)
            ejemplos += lote.size(0)

        perdida_train = suma / ejemplos

        modelo_inicial.eval()
        suma_val = 0.0
        ejemplos_val = 0

        with torch.no_grad():
            for (lote,) in val_loader:
                lote = lote.to(DEVICE)
                with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
                    _, perdida = modelo_inicial(lote, lote)
                suma_val += perdida.item() * lote.size(0)
                ejemplos_val += lote.size(0)

        perdida_val = suma_val / ejemplos_val

        historial["train"].append(perdida_train)
        historial["val"].append(perdida_val)

        if perdida_val < mejor_val:
            mejor_val = perdida_val
            mejor_epoca = epoca + 1
            mejor_estado = {}
            for nombre, parametro in modelo_inicial.state_dict().items():
                mejor_estado[nombre] = parametro.detach().cpu().clone()

        print("Epoca", epoca + 1,
              "| train:", round(perdida_train, 3),
              "| val:", round(perdida_val, 3),
              "| lr:", round(scheduler.get_last_lr()[0], 7),
              "| tiempo:", round(time.time() - inicio, 1), "s")

    print("Mejor epoca:", mejor_epoca, "| mejor val:", round(mejor_val, 3))
else:
    print("Cargado desde Drive, mejor epoca:", mejor_epoca)


Graficamos las curvas de pérdida para ver si hay sobreajuste.


In [ ]:
if len(historial["val"]) > 0:
    plt.figure(figsize=(7, 4))
    plt.plot(range(1, len(historial["train"]) + 1), historial["train"], marker="o", label="train")
    plt.plot(range(1, len(historial["val"]) + 1), historial["val"], marker="o", label="validacion")
    plt.title("Perdida por epoca - modelo desde cero")
    plt.xlabel("epoca")
    plt.ylabel("cross-entropy")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()
else:
    print("Error no hay data para la grafica")


Guardamos el mejor modelo en Drive con su historial y liberamos la memoria de la GPU.


In [ ]:
if entreno_scratch:
    modelo_inicial.load_state_dict(mejor_estado)

    torch.save(modelo_inicial.state_dict(), ruta_scratch)

    archivo = open(ruta_historial_scratch, "w")
    json.dump({"historial": historial, "mejor_val": mejor_val, "mejor_epoca": mejor_epoca}, archivo)
    archivo.close()

    print("Checkpoint:", ruta_scratch)
    print("Epoca guardada:", mejor_epoca)
    print("Tamano:", round(os.path.getsize(ruta_scratch) / 1024**2, 1), "MB")
else:
    print("Checkpoint:", ruta_scratch)
    print("Epoca guardada:", mejor_epoca)

del modelo_inicial
del optimizer
del scheduler
liberar_vram_total()


## 7. Finetuning de los modelos pre-entrenados

Ajustamos SmolLM2-135M y SmolLM2-360M con **full finetuning**: se actualizan el 100 % de los
parámetros, sin LoRA ni QLoRA. Para que quepan en la GPU usamos la tríada del anexo: precisión
mixta, acumulación de gradientes y gradient checkpointing. Cada modelo se entrena 2 épocas y se
guarda la mejor época según la pérdida de validación.


Definimos los hiperparámetros de cada finetuning.

Usamos los learning rate que sugiere la tabla del enunciado: 5e-5 para el 135M y 3e-5 para el
360M, porque con solo 2 épocas no conviene mover mucho los pesos que ya trae el
preentrenamiento. Entrenamos 2 épocas porque el 135M pasó de 1,594 a 1,580 y la tercera ya
subía a 1,578, y el 360M pasó de 1,383 a 1,376 y casi no mejoraba. El micro-batch de 8 con
acumulación de 2 da el batch efectivo de 16 que pide el enunciado, y en la L4 sobra VRAM:
el 135M marcó 4,88 GB y el 360M 8,44 GB de 22 GB. Con el micro-batch de 2 que sugiere el
anexo el 360M tardaba 1.972 segundos por época y con 8 tarda unos 590. El weight decay de
0,01 es una regularización suave, porque 0,1 es la del preentrenamiento y aquí frenaría la
adaptación al dominio. El calentamiento de 50 pasos es el 7 % de los 673 pasos por época.
El gradient checkpointing con `use_cache` en `False` es lo que exige el anexo técnico.
Guardamos el tokenizador junto a cada checkpoint e imprimimos el porcentaje entrenable.


In [ ]:
EPOCAS_FT = 2
CALENTAMIENTO_FT = 50
WEIGHT_DECAY_FT = 0.01

config_135m = {
    "nombre": "SmolLM2-135M",
    "modelo": SMOL_135M,
    "revision": SMOL_135M_REVISION,
    "lr": 5e-5,
    "micro_batch": 8,
    "grad_accum": 2,
    "carpeta": "checkpoint_smollm2_135m_ft",
}

config_360m = {
    "nombre": "SmolLM2-360M",
    "modelo": SMOL_360M,
    "revision": SMOL_360M_REVISION,
    "lr": 3e-5,
    "micro_batch": 8,
    "grad_accum": 2,
    "carpeta": "checkpoint_smollm2_360m_ft",
}

print("135M: lr", config_135m["lr"], "| micro", config_135m["micro_batch"], "x", config_135m["grad_accum"])
print("360M: lr", config_360m["lr"], "| micro", config_360m["micro_batch"], "x", config_360m["grad_accum"])
print("Epocas:", EPOCAS_FT)


Definimos la función que ajusta un modelo y guarda la mejor época.


In [ ]:
def ajustar_modelo(config):
    ruta_guardado = os.path.join(DRIVE_DIR, config["carpeta"])
    ruta_temporal = ruta_guardado + "_mejor"
    ruta_estado = ruta_guardado + "_estado.json"

    if os.path.exists(ruta_guardado):
        print(config["nombre"], ": ya hay checkpoint, no se entrena el modelo")
        historial = {"train": [], "val": []}
        mejor_val = float("nan")
        mejor_epoca = 0
        if os.path.exists(ruta_estado):
            archivo = open(ruta_estado, "r")
            guardado = json.load(archivo)
            archivo.close()
            historial = guardado["historial"]
            mejor_val = guardado["mejor_val"]
            mejor_epoca = guardado["mejor_epoca"]
            print("historial recuperado de", ruta_estado)
        return {
            "historial": historial,
            "mejor_val": mejor_val,
            "mejor_epoca": mejor_epoca,
            "segundos": 0,
            "omitido": True,
        }

    historial = {"train": [], "val": []}
    mejor_val = float("inf")
    mejor_epoca = 0
    epoca_inicio = 0

    if os.path.exists(ruta_estado):
        archivo = open(ruta_estado, "r")
        guardado = json.load(archivo)
        archivo.close()
        historial = guardado["historial"]
        mejor_val = guardado["mejor_val"]
        mejor_epoca = guardado["mejor_epoca"]
        epoca_inicio = guardado["epoca_completada"]
        print(config["nombre"], ": se retoma desde la epoca", epoca_inicio + 1)

    if epoca_inicio == 0:
        modelo = AutoModelForCausalLM.from_pretrained(
            config["modelo"],
            revision=config["revision"],
        )
    else:
        ruta_ultima = ruta_guardado + "_epoca" + str(epoca_inicio)
        if os.path.exists(ruta_ultima):
            modelo = AutoModelForCausalLM.from_pretrained(ruta_ultima)
            print(config["nombre"], ": pesos recuperados de", ruta_ultima)
        else:
            modelo = AutoModelForCausalLM.from_pretrained(
                config["modelo"],
                revision=config["revision"],
            )

    modelo.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    modelo.config.use_cache = False
    modelo.to(DEVICE)

    total_parametros = 0
    entrenables = 0
    for parametro in modelo.parameters():
        total_parametros += parametro.numel()
        if parametro.requires_grad:
            entrenables += parametro.numel()

    print(config["nombre"], "- parametros:", total_parametros)
    print(config["nombre"], "- entrenables:", round(entrenables / total_parametros * 100, 1), "%")

    datos = TensorDataset(X_train)

    pasos_por_epoca = math.ceil(len(datos) / config["micro_batch"] / config["grad_accum"])
    pasos_totales = pasos_por_epoca * EPOCAS_FT

    optimizador = torch.optim.AdamW(
        modelo.parameters(),
        lr=config["lr"],
        betas=(0.9, 0.95),
        weight_decay=WEIGHT_DECAY_FT,
    )
    escalador = torch.amp.GradScaler("cuda", enabled=USE_SCALER)

    def tasa(paso):
        if paso < CALENTAMIENTO_FT:
            return (paso + 1) / CALENTAMIENTO_FT
        avance = (paso - CALENTAMIENTO_FT) / (pasos_totales - CALENTAMIENTO_FT)
        if avance > 1:
            avance = 1
        return 0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * avance))

    planificador = torch.optim.lr_scheduler.LambdaLR(optimizador, tasa)

    inicio_modelo = time.time()

    for epoca_num in range(epoca_inicio, EPOCAS_FT):
        epoca = epoca_num + 1
        generador = torch.Generator()
        generador.manual_seed(SEED + epoca)
        loader = DataLoader(datos, batch_size=config["micro_batch"], shuffle=True, generator=generador)
        modelo.train()
        suma = 0.0
        ejemplos = 0
        paso = 0
        optimizador.zero_grad(set_to_none=True)
        inicio = time.time()

        for (lote,) in loader:
            lote = lote.to(DEVICE)

            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
                salida = modelo(input_ids=lote, labels=lote)
                perdida = salida.loss / config["grad_accum"]

            escalador.scale(perdida).backward()
            suma += salida.loss.item() * lote.size(0)
            ejemplos += lote.size(0)
            paso += 1

            if paso % config["grad_accum"] == 0:
                escalador.unscale_(optimizador)
                torch.nn.utils.clip_grad_norm_(modelo.parameters(), CLIP_GRAD_NORM)
                escalador.step(optimizador)
                escalador.update()
                optimizador.zero_grad(set_to_none=True)
                planificador.step()

        if paso % config["grad_accum"] != 0:
            escalador.unscale_(optimizador)
            torch.nn.utils.clip_grad_norm_(modelo.parameters(), CLIP_GRAD_NORM)
            escalador.step(optimizador)
            escalador.update()
            optimizador.zero_grad(set_to_none=True)
            planificador.step()

        perdida_train = suma / ejemplos

        modelo.eval()
        suma_val = 0.0
        ejemplos_val = 0

        with torch.no_grad():
            for (lote,) in val_loader:
                lote = lote.to(DEVICE)
                with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
                    salida = modelo(input_ids=lote, labels=lote)
                suma_val += salida.loss.item() * lote.size(0)
                ejemplos_val += lote.size(0)

        perdida_val = suma_val / ejemplos_val
        historial["train"].append(perdida_train)
        historial["val"].append(perdida_val)

        if perdida_val < mejor_val:
            mejor_val = perdida_val
            mejor_epoca = epoca
            modelo.save_pretrained(ruta_temporal)
            print("  mejor hasta ahora, guardado en", ruta_temporal)

        ruta_epoca = ruta_guardado + "_epoca" + str(epoca)
        modelo.save_pretrained(ruta_epoca)

        archivo = open(ruta_estado, "w")
        json.dump({"epoca_completada": epoca, "historial": historial, "mejor_val": mejor_val, "mejor_epoca": mejor_epoca}, archivo)
        archivo.close()

        print(config["nombre"],
              "- epoca", epoca,
              "| train:", round(perdida_train, 3),
              "| val:", round(perdida_val, 3),
              "| lr:", round(planificador.get_last_lr()[0], 8),
              "| tiempo:", round(time.time() - inicio, 1), "s")

    segundos = time.time() - inicio_modelo
    print(config["nombre"], "- VRAM pico:", round(torch.cuda.max_memory_allocated() / 1024**3, 2), "GB")

    del optimizador
    del planificador

    if mejor_epoca != EPOCAS_FT:
        print(config["nombre"], ": se recarga la mejor epoca", mejor_epoca)
        del modelo
        liberar_vram_total()
        modelo = AutoModelForCausalLM.from_pretrained(ruta_temporal)
        modelo.to(DEVICE)

    modelo.save_pretrained(ruta_guardado)
    tokenizer.save_pretrained(ruta_guardado)
    print(config["nombre"], ": checkpoint final en", ruta_guardado)

    if os.path.exists(ruta_temporal):
        shutil.rmtree(ruta_temporal)

    for epoca_borrar in range(1, EPOCAS_FT + 1):
        ruta_borrar = ruta_guardado + "_epoca" + str(epoca_borrar)
        if os.path.exists(ruta_borrar):
            shutil.rmtree(ruta_borrar)

    del modelo
    liberar_vram_total()

    return {
        "historial": historial,
        "mejor_val": mejor_val,
        "mejor_epoca": mejor_epoca,
        "segundos": segundos,
        "omitido": False,
    }


Finetuneamos SmolLM2-135M.


In [ ]:
resultado_135m = ajustar_modelo(config_135m)


Liberamos la memoria de la GPU antes de cargar el modelo de 360M.


In [ ]:
liberar_vram_total()
print("VRAM en uso:", round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")


Finetuneamos SmolLM2-360M.


In [ ]:
resultado_360m = ajustar_modelo(config_360m)


Graficamos las curvas de validación de los dos finetunings.


In [ ]:
plt.figure(figsize=(7, 4))

if len(resultado_135m["historial"]["val"]) > 0:
    plt.plot(range(1, len(resultado_135m["historial"]["val"]) + 1), resultado_135m["historial"]["val"], marker="o", label="135M val")
if len(resultado_360m["historial"]["val"]) > 0:
    plt.plot(range(1, len(resultado_360m["historial"]["val"]) + 1), resultado_360m["historial"]["val"], marker="o", label="360M val")

plt.title("Perdida de validacion durante el finetuning")
plt.xlabel("epoca")
plt.ylabel("cross-entropy")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


Resumimos la mejor época y la perplejidad de cada modelo.


In [ ]:
resultados = [("SmolLM2-135M", resultado_135m), ("SmolLM2-360M", resultado_360m)]

for nombre, resultado in resultados:
    if resultado["omitido"]:
        print(nombre, ": se uso el checkpoint existente")
    print(nombre,
          "| mejor epoca:", resultado["mejor_epoca"],
          "| val:", round(resultado["mejor_val"], 3),
          "| PPL:", round(math.exp(resultado["mejor_val"]), 2),
          "| minutos:", round(resultado["segundos"] / 60, 1))


## 8. Evaluación y comparación

Medimos las cinco configuraciones con el mismo tokenizador: perplejidad global sobre los 1.365 bloques
de prueba y validez sintáctica con `ast` sobre 200 ejemplos. Los modelos base se evalúan sin
entrenar, tal como lo pide el enunciado, y cada modelo se carga solo mientras se mide.


Definimos los hiperparámetros de la evaluación.

Evaluamos 200 ejemplos porque es lo que pide el enunciado, y los elegimos con la semilla 7
para que la submuestra sea reproducible y distinta del split. El prompt fijo de 64 tokens
permite generar en lotes sin padding y son unas 5 líneas de contexto. Generamos 128 tokens
nuevos porque ahí cabe una función completa. La temperatura de 0,7 con top-k de 50 da variedad
sin disparates. Los lotes de 8 para PPL y de 20 para generación caben en los bloques de 256
incluso en el 360M. La PPL se mide sobre los 1.365 bloques que salen de los 800 archivos de
prueba.


In [ ]:
N_AST = 200
SEMILLA_AST = 7
LARGO_PROMPT = 64
NUEVOS = 128
TEMP = 0.7
TOP_K = 50
LOTE_EVAL = 8
LOTE_GEN = 20

print("Ejemplos AST:", N_AST)
print("Prompt:", LARGO_PROMPT, "| nuevos:", NUEVOS)
print("Temperatura:", TEMP, "| top-k:", TOP_K)


Implementamos el bucle autorregresivo propio con temperatura y top-k.


In [ ]:
def generar_texto(modelo, inicio, nuevos, temperatura, top_k, semilla):
    modelo.eval()
    ids = inicio.clone()
    gen = torch.Generator(device=DEVICE)
    gen.manual_seed(semilla)
    es_scratch = isinstance(modelo, ModeloInicial)
    termino_eos = False
    with torch.no_grad():
        pasado = None
        for paso in range(nuevos):
            if es_scratch:
                entrada = ids[:, -CTX:]
                with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
                    salida = modelo(entrada)
                logits = salida[0]
            else:
                if paso == 0:
                    entrada = ids[:, -CTX:]
                    with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
                        salida = modelo(entrada, use_cache=True)
                    pasado = salida.past_key_values
                    logits = salida.logits
                else:
                    with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
                        salida = modelo(siguiente, past_key_values=pasado, use_cache=True)
                    pasado = salida.past_key_values
                    logits = salida.logits
            ultimos = logits[:, -1, :].float() / temperatura
            if top_k > 0:
                valores, indices = torch.topk(ultimos, top_k)
                minimo = valores[:, -1].unsqueeze(1)
                relleno = torch.tensor(float("-inf"), device=ultimos.device)
                ultimos = torch.where(ultimos < minimo, relleno, ultimos)
            probs = F.softmax(ultimos, dim=-1)
            siguiente = torch.multinomial(probs, num_samples=1, generator=gen)
            ids = torch.cat([ids, siguiente], dim=1)
            if (siguiente == tokenizer.eos_token_id).all():
                termino_eos = True
                break
    return ids, termino_eos


Calculamos la perplejidad global acumulando por token.


In [ ]:
def calcular_ppl(modelo, datos, lote_eval):
    modelo.eval()
    suma = 0.0
    total = 0
    with torch.no_grad():
        for inicio in range(0, len(datos), lote_eval):
            lote = datos[inicio:inicio + lote_eval].to(DEVICE)
            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
                salida = modelo(lote)
            if isinstance(salida, tuple):
                logits = salida[0]
            else:
                logits = salida.logits
            planos = logits[:, :-1, :].reshape(-1, VOCAB_SIZE)
            objetivos = lote[:, 1:].reshape(-1)
            suma += F.cross_entropy(planos, objetivos, reduction="sum").item()
            total += objetivos.numel()
    return math.exp(suma / total)


Definimos dos ayudas: la verificación sintáctica y el intervalo de Wilson al 95 %.


In [ ]:
def es_valido(codigo):
    try:
        ast.parse(codigo)
        return True
    except SyntaxError:
        return False
    except ValueError:
        return False
    except RecursionError:
        return False


def intervalo_wilson(buenas, total):
    z = 1.96
    p = buenas / total
    denominador = 1 + z * z / total
    centro = (p + z * z / (2 * total)) / denominador
    margen = z * math.sqrt(p * (1 - p) / total + z * z / (4 * total * total)) / denominador
    return (centro - margen) * 100, (centro + margen) * 100


Medimos la validez sintáctica generando continuaciones y verificando prompt más continuación.


In [ ]:
def evaluar_ast(modelo, textos_base, indices, lote_gen, semilla):
    validas = 0
    for inicio in range(0, len(indices), lote_gen):
        grupo = indices[inicio:inicio + lote_gen]
        prompts = []
        for j in grupo:
            ids = tokenizer.encode(textos_base[j], add_special_tokens=False)[:LARGO_PROMPT]
            while len(ids) < LARGO_PROMPT:
                ids.append(tokenizer.eos_token_id)
            prompts.append(ids)
        entrada = torch.tensor(prompts, dtype=torch.long, device=DEVICE)
        generados, termino_eos = generar_texto(modelo, entrada, NUEVOS, TEMP, TOP_K, semilla + inicio)
        for f in range(generados.size(0)):
            texto = tokenizer.decode(generados[f].tolist())
            if termino_eos == False:
                lineas = texto.split("\n")
                texto = "\n".join(lineas[:-1])
            if es_valido(texto):
                validas += 1
    return validas


Elegimos los 200 ejemplos de prueba con su propia semilla.


In [ ]:
rng_ast = random.Random(SEMILLA_AST)
indices_ast = rng_ast.sample(range(len(textos_test)), N_AST)

print("Ejemplos de prueba:", len(textos_test))
print("Submuestra AST:", len(indices_ast))
print("Primeros indices:", indices_ast[:5])


Evaluamos las cinco configuraciones cargando un modelo a la vez.


In [ ]:
nombres = ["scratch", "135M base", "135M FT", "360M base", "360M FT"]
ppls = []
porcentajes = []
parametros_lista = []
validas_lista = []
inf_lista = []
sup_lista = []

for indice in range(len(nombres)):
    nombre = nombres[indice]
    if nombre == "scratch":
        modelo = ModeloInicial().to(DEVICE)
        ruta = os.path.join(DRIVE_DIR, "checkpoint_scratch.pt")
        modelo.load_state_dict(torch.load(ruta, map_location=DEVICE))
    elif nombre == "135M base":
        modelo = AutoModelForCausalLM.from_pretrained(SMOL_135M, revision=SMOL_135M_REVISION)
        modelo.to(DEVICE)
    elif nombre == "135M FT":
        ruta = os.path.join(DRIVE_DIR, "checkpoint_smollm2_135m_ft")
        modelo = AutoModelForCausalLM.from_pretrained(ruta)
        modelo.to(DEVICE)
    elif nombre == "360M base":
        modelo = AutoModelForCausalLM.from_pretrained(SMOL_360M, revision=SMOL_360M_REVISION)
        modelo.to(DEVICE)
    else:
        ruta = os.path.join(DRIVE_DIR, "checkpoint_smollm2_360m_ft")
        modelo = AutoModelForCausalLM.from_pretrained(ruta)
        modelo.to(DEVICE)

    if nombre != "scratch":
        modelo.config.use_cache = True

    total_parametros = 0
    for parametro in modelo.parameters():
        total_parametros += parametro.numel()
    parametros_lista.append(total_parametros)

    inicio = time.time()
    ppl = calcular_ppl(modelo, X_test, LOTE_EVAL)
    validas = evaluar_ast(modelo, textos_test, indices_ast, LOTE_GEN, SEED)
    pct = validas / N_AST * 100
    inf, sup = intervalo_wilson(validas, N_AST)

    ppls.append(ppl)
    porcentajes.append(pct)
    validas_lista.append(validas)
    inf_lista.append(inf)
    sup_lista.append(sup)

    print(nombre, "| params:", total_parametros, "| PPL:", round(ppl, 2),
          "| AST:", round(pct, 1), "% | IC95:", round(inf, 1), "-", round(sup, 1),
          "| tiempo:", round(time.time() - inicio, 1), "s")

    del modelo
    liberar_vram_total()

print("VRAM en uso:", round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")


Medimos el techo de referencia con la continuación real de cada archivo.


In [ ]:
validas_ref = 0
completos_validos = 0

for j in indices_ast:
    ids_completos = tokenizer.encode(textos_test[j], add_special_tokens=False)
    ids_prompt = ids_completos[:LARGO_PROMPT]
    while len(ids_prompt) < LARGO_PROMPT:
        ids_prompt.append(tokenizer.eos_token_id)
    resto = ids_completos[LARGO_PROMPT:]
    truncada = len(resto) > NUEVOS
    texto = tokenizer.decode(ids_prompt + resto[:NUEVOS])
    if truncada:
        lineas = texto.split("\n")
        texto = "\n".join(lineas[:-1])
    if es_valido(texto):
        validas_ref += 1
    if es_valido(textos_test[j]):
        completos_validos += 1

techo_ref = validas_ref / N_AST * 100
techo_completo = completos_validos / N_AST * 100

print("Continuacion real con el mismo protocolo:", validas_ref, "de", N_AST, "=", round(techo_ref, 1), "%")
print("Archivos completos validos:", completos_validos, "de", N_AST, "=", round(techo_completo, 1), "%")


Mostramos la tabla comparativa de las cinco configuraciones.


In [ ]:
print("Configuracion | Parametros | PPL test | AST valido % | IC95")
for i in range(len(nombres)):
    print(nombres[i], "|", parametros_lista[i], "|", round(ppls[i], 2), "|", round(porcentajes[i], 1),
          "|", round(inf_lista[i], 1), "-", round(sup_lista[i], 1))


Graficamos las perplejidades para comparar los cinco modelos.


In [ ]:
plt.figure(figsize=(7, 4))
plt.bar(nombres, ppls)
plt.yscale("log")
plt.title("Perplejidad en prueba por configuracion (escala log)")
plt.ylabel("PPL")
plt.xticks(rotation=15)
for i in range(len(nombres)):
    plt.text(i, ppls[i] * 1.05, round(ppls[i], 2), ha="center")
plt.grid(axis="y", alpha=0.3)
plt.show()


Graficamos el porcentaje de código válido de cada configuración.


In [ ]:
errores_bajo = []
errores_alto = []
for i in range(len(nombres)):
    errores_bajo.append(porcentajes[i] - inf_lista[i])
    errores_alto.append(sup_lista[i] - porcentajes[i])

plt.figure(figsize=(7, 4))
plt.bar(nombres, porcentajes, yerr=[errores_bajo, errores_alto], capsize=5)
plt.axhline(techo_ref, color="black", linestyle="--", label="continuacion real")
plt.title("Codigo valido por configuracion (n = 200, IC95)")
plt.ylabel("% valido")
plt.xticks(rotation=15)
plt.legend()
plt.grid(axis="y", alpha=0.3)
plt.show()


Revisamos el sobreajuste comparando la pérdida de entrenamiento con la de validación.


In [ ]:
ruta_historial_scratch = os.path.join(DRIVE_DIR, "historial_scratch.json")
archivo = open(ruta_historial_scratch, "r")
guardado_scratch = json.load(archivo)
archivo.close()

ruta_estado_135m = os.path.join(DRIVE_DIR, "checkpoint_smollm2_135m_ft_estado.json")
archivo = open(ruta_estado_135m, "r")
guardado_135m = json.load(archivo)
archivo.close()

ruta_estado_360m = os.path.join(DRIVE_DIR, "checkpoint_smollm2_360m_ft_estado.json")
archivo = open(ruta_estado_360m, "r")
guardado_360m = json.load(archivo)
archivo.close()

print("scratch: train", round(guardado_scratch["historial"]["train"][-1], 3),
      "| val", round(guardado_scratch["historial"]["val"][-1], 3))
print("135M FT: train", round(guardado_135m["historial"]["train"][-1], 3),
      "| val", round(guardado_135m["historial"]["val"][-1], 3))
print("360M FT: train", round(guardado_360m["historial"]["train"][-1], 3),
      "| val", round(guardado_360m["historial"]["val"][-1], 3))


## 9. Generación de ejemplos

Generamos continuaciones con nuestra propia función autorregresiva para los tres modelos
entrenados. Usamos los mismos 3 prompts de prueba para los tres, con semilla fija por modelo
y por ejemplo, para que la comparación sea justa y reproducible.


Definimos 3 prompts de código limpio, iguales para los tres modelos.


In [ ]:
prompts_ejemplos = [
    "def fibonacci(n):\n    ",
    "import os\nimport json\n\n\ndef load_config(path):\n    ",
    "class Stack(object):\n\n    def __init__(self):\n        ",
]

ids_ejemplos = []
for texto_prompt in prompts_ejemplos:
    ids = tokenizer.encode(texto_prompt, add_special_tokens=False)
    ids_ejemplos.append(ids)

for k in range(len(prompts_ejemplos)):
    print("Prompt", k + 1, ":")
    print(prompts_ejemplos[k])


Generamos 3 continuaciones con el modelo desde cero.


In [ ]:
modelo = ModeloInicial().to(DEVICE)
ruta = os.path.join(DRIVE_DIR, "checkpoint_scratch.pt")
modelo.load_state_dict(torch.load(ruta, map_location=DEVICE))

ejemplos_scratch = []
largos_scratch = []

for p in range(len(ids_ejemplos)):
    entrada = torch.tensor([ids_ejemplos[p]], dtype=torch.long, device=DEVICE)
    generados, fin = generar_texto(modelo, entrada, NUEVOS, TEMP, TOP_K, SEED + p)
    texto = tokenizer.decode(generados[0].tolist())
    ejemplos_scratch.append(texto)
    largos_scratch.append(generados.size(1) - len(ids_ejemplos[p]))
    print("Ejemplo", p + 1, "| tokens nuevos:", generados.size(1) - len(ids_ejemplos[p]))
    print(texto)
    print()

del modelo
liberar_vram_total()


Generamos las mismas 3 continuaciones con SmolLM2-135M finetuneado.


In [ ]:
ruta = os.path.join(DRIVE_DIR, "checkpoint_smollm2_135m_ft")
modelo = AutoModelForCausalLM.from_pretrained(ruta)
modelo.to(DEVICE)
modelo.config.use_cache = True

ejemplos_135m = []
largos_135m = []

for p in range(len(ids_ejemplos)):
    entrada = torch.tensor([ids_ejemplos[p]], dtype=torch.long, device=DEVICE)
    generados, fin = generar_texto(modelo, entrada, NUEVOS, TEMP, TOP_K, SEED + 10 + p)
    texto = tokenizer.decode(generados[0].tolist())
    ejemplos_135m.append(texto)
    largos_135m.append(generados.size(1) - len(ids_ejemplos[p]))
    print("Ejemplo", p + 1, "| tokens nuevos:", generados.size(1) - len(ids_ejemplos[p]))
    print(texto)
    print()

del modelo
liberar_vram_total()


Generamos las mismas 3 continuaciones con SmolLM2-360M finetuneado.


In [ ]:
ruta = os.path.join(DRIVE_DIR, "checkpoint_smollm2_360m_ft")
modelo = AutoModelForCausalLM.from_pretrained(ruta)
modelo.to(DEVICE)
modelo.config.use_cache = True

ejemplos_360m = []
largos_360m = []

for p in range(len(ids_ejemplos)):
    entrada = torch.tensor([ids_ejemplos[p]], dtype=torch.long, device=DEVICE)
    generados, fin = generar_texto(modelo, entrada, NUEVOS, TEMP, TOP_K, SEED + 20 + p)
    texto = tokenizer.decode(generados[0].tolist())
    ejemplos_360m.append(texto)
    largos_360m.append(generados.size(1) - len(ids_ejemplos[p]))
    print("Ejemplo", p + 1, "| tokens nuevos:", generados.size(1) - len(ids_ejemplos[p]))
    print(texto)
    print()

del modelo
liberar_vram_total()
print("VRAM en uso:", round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")


Resumimos cuántos tokens generó cada modelo en cada ejemplo.


In [ ]:
nombres_gen = ["scratch", "135M FT", "360M FT"]
largos_todos = [largos_scratch, largos_135m, largos_360m]

for m in range(len(nombres_gen)):
    for p in range(3):
        tope = "si" if largos_todos[m][p] == NUEVOS else "no"
        print(nombres_gen[m], "| ejemplo", p + 1, "| tokens:", largos_todos[m][p], "| llego al tope:", tope)
